<a href="https://colab.research.google.com/github/abdul2924/abdul2924/blob/main/Assignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [28]:
# Initial Setup and Project Structure
!apt-get update -qq > /dev/null
!apt-get install -y openjdk-8-jdk-headless -qq > /dev/null
!pip install pyspark==3.5.0 pyarrow -q

import os

os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"


from google.colab import drive
drive.mount('/content/drive', force_remount=True)


BASE_PATH = "/content/drive/MyDrive/yellow_tripdata_2019"
RAW_PATH = f"{BASE_PATH}/data/raw"
PROCESSED_PATH = f"{BASE_PATH}/data/processed/clean_taxi_data"


for folder in ["notebooks", "data/raw", "data/processed", "tableau", "config", "scripts", "tests"]:
    os.makedirs(os.path.join(BASE_PATH, folder), exist_ok=True)

print(f" Project mapped to: {BASE_PATH}")

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Mounted at /content/drive
 Project mapped to: /content/drive/MyDrive/yellow_tripdata_2019


In [29]:
# Initialize Spark Session
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("NYC_Taxi_2019_Ingestion") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.sql.execution.arrow.pyspark.enabled", "true") \
    .config("spark.driver.memory", "8g") \
    .config("spark.executor.memory", "6g") \
    .config("spark.sql.parquet.mergeSchema", "false") \
    .getOrCreate()

print(" Spark Session Active")
print(f"Version: {spark.version}")

 Spark Session Active
Version: 3.5.0


In [30]:
# Load Raw Data and Perform Initial Validation
from pyspark.sql.functions import col, isnan, when, count, date_format
from pyspark.sql.types import NumericType


print(f"Reading from: {RAW_PATH}")

df_raw = spark.read.parquet(f"{RAW_PATH}/yellow_2019_*.parquet")
total_rows = df_raw.count()
print(f"Total Raw Rows: {total_rows:,}")


check_cols = ["fare_amount", "trip_distance", "passenger_count", "tpep_pickup_datetime"]
null_exprs = []

for c in check_cols:

    if isinstance(df_raw.schema[c].dataType, NumericType):
        null_exprs.append(count(when(col(c).isNull() | isnan(col(c)), c)).alias(f"{c}_nulls"))
    else:
        null_exprs.append(count(when(col(c).isNull(), c)).alias(f"{c}_nulls"))

print("\n Null Value Summary ")
df_raw.select(*null_exprs).show()

print("\n Fare Amount Summary ")
df_raw.select("fare_amount").summary("count", "min", "mean", "max").show()

Reading from: /content/drive/MyDrive/yellow_tripdata_2019/data/raw
Total Raw Rows: 84,598,444

 Null Value Summary 
+-----------------+-------------------+---------------------+--------------------------+
|fare_amount_nulls|trip_distance_nulls|passenger_count_nulls|tpep_pickup_datetime_nulls|
+-----------------+-------------------+---------------------+--------------------------+
|                0|                  0|               444383|                         0|
+-----------------+-------------------+---------------------+--------------------------+


 Fare Amount Summary 
+-------+-----------------+
|summary|      fare_amount|
+-------+-----------------+
|  count|         84598444|
|    min|          -1856.0|
|   mean|13.41263973283576|
|    max|         943274.8|
+-------+-----------------+



In [31]:
# List Contents of Raw Data Directory
import os


raw_data_dir = RAW_PATH

if os.path.exists(raw_data_dir):
    print(f"Contents of {raw_data_dir}:")
    for item in os.listdir(raw_data_dir):
        print(item)
else:
    print(f"The directory {raw_data_dir} does not exist.")

Contents of /content/drive/MyDrive/yellow_tripdata_2019/data/raw:
yellow_2019_01.parquet
yellow_2019_02.parquet
yellow_2019_03.parquet
yellow_2019_04.parquet
yellow_2019_05.parquet
yellow_2019_07.parquet
yellow_2019_06.parquet
yellow_2019_08.parquet
yellow_2019_09.parquet
yellow_2019_10.parquet
yellow_2019_11.parquet
yellow_2019_12.parquet


In [32]:
# Clean and Filter Data
from pyspark.sql.functions import col, date_format, year


df_clean = df_raw.filter(
    (col("fare_amount").between(1.0, 500.0)) &
    (col("trip_distance").between(0.1, 100.0)) &
    (col("passenger_count").cast("int").between(1, 6)) &
    (year(col("tpep_pickup_datetime")) == 2019)
).dropDuplicates()


df_clean = df_clean.withColumn("pickup_month", date_format(col("tpep_pickup_datetime"), "yyyy-MM"))

print(f"Cleaned Data Rows: {df_clean.count():,}")

Cleaned Data Rows: 81,581,307


In [33]:
# Write Cleaned Data to Processed Path
(df_clean.coalesce(50)
 .write
 .partitionBy("pickup_month")
 .mode("overwrite")
 .option("compression", "snappy")
 .parquet(PROCESSED_PATH))

print(f" Data successfully written to: {PROCESSED_PATH}")

 Data successfully written to: /content/drive/MyDrive/yellow_tripdata_2019/data/processed/clean_taxi_data


In [34]:
# Verify Partition Distribution of Processed Data
df_final = spark.read.parquet(PROCESSED_PATH)

print("--- Partition Distribution ( only 2019 data) ")
df_final.groupBy("pickup_month").count().orderBy("pickup_month").show(12)

--- Partition Distribution ( only 2019 data) 
+------------+-------+
|pickup_month|  count|
+------------+-------+
|     2019-01|7476044|
|     2019-02|6830367|
|     2019-03|7616189|
|     2019-04|7223713|
|     2019-05|7340317|
|     2019-06|6724242|
|     2019-07|6068631|
|     2019-08|5835523|
|     2019-09|6315105|
|     2019-10|6930694|
|     2019-11|6603163|
|     2019-12|6617319|
+------------+-------+



feature eng

In [35]:
# Define and Execute Feature Engineering Pipeline
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.ml import Pipeline, Transformer
from pyspark.ml.feature import VectorAssembler, StandardScaler, OneHotEncoder, StringIndexer
from pyspark.ml.util import DefaultParamsReadable, DefaultParamsWritable


BASE_PATH = "/content/drive/MyDrive/yellow_tripdata_2019"
INPUT_DATA = f"{BASE_PATH}/data/processed/clean_taxi_data"
OUTPUT_PATH = f"{BASE_PATH}/data/final"


spark = SparkSession.builder \
    .appName("NYCTaxi_FeatureEngineering_2019") \
    .config("spark.sql.parquet.enableVectorizedReader", "true") \
    .getOrCreate()


df = spark.read.parquet(INPUT_DATA)


class TaxiFeatureGenerator(Transformer, DefaultParamsReadable, DefaultParamsWritable):

    def _transform(self, dataset):

        R = 6371.0

        dlat = F.radians(F.col("pickup_latitude") - F.col("dropoff_latitude"))
        dlon = F.radians(F.col("pickup_longitude") - F.col("dropoff_longitude"))

        a = (F.sin(dlat / 2) ** 2 +
             F.cos(F.radians(F.col("pickup_latitude"))) *
             F.cos(F.radians(F.col("dropoff_latitude"))) *
             F.sin(dlon / 2) ** 2)

        c = 2 * F.atan2(F.sqrt(a), F.sqrt(1 - a))
        haversine_km = R * c

        return dataset.withColumn("haversine_km", haversine_km.cast(DoubleType())) \
            .withColumn("trip_duration_min",
                (F.unix_timestamp("tpep_dropoff_datetime") - F.unix_timestamp("tpep_pickup_datetime")) / 60.0) \
            .withColumn("pickup_hour", F.hour("tpep_pickup_datetime")) \
            .withColumn("day_of_week", F.dayofweek("tpep_pickup_datetime")) \
            .withColumn("is_peak_hour", F.when(F.col("pickup_hour").between(7,9) |
                                             F.col("pickup_hour").between(16,19), 1).otherwise(0)) \
            .withColumn("is_weekend", F.when(F.col("day_of_week").isin(1, 7), 1).otherwise(0)) \
            .fillna(0) # Proper handling of nulls


df_prepared = df.withColumn("pickup_latitude", F.lit(40.7128)) \
                .withColumn("pickup_longitude", F.lit(-74.0060)) \
                .withColumn("dropoff_latitude", F.lit(40.7128)) \
                .withColumn("dropoff_longitude", F.lit(-74.0060))

feature_gen = TaxiFeatureGenerator()


indexer = StringIndexer(inputCol="day_of_week", outputCol="dow_index")
encoder = OneHotEncoder(inputCol="dow_index", outputCol="dow_vector")


numeric_features = [
    "passenger_count", "trip_distance", "haversine_km",
    "trip_duration_min", "pickup_hour", "is_peak_hour", "is_weekend"
]

assembler = VectorAssembler(
    inputCols=numeric_features + ["dow_vector"],
    outputCol="features_unscaled"
)

scaler = StandardScaler(
    inputCol="features_unscaled",
    outputCol="features",
    withStd=True,
    withMean=True
)


pipeline = Pipeline(stages=[feature_gen, indexer, encoder, assembler, scaler])


pipeline_model = pipeline.fit(df_prepared)
df_final = pipeline_model.transform(df_prepared)


train_df = df_final.filter(F.col("pickup_month") <= "2019-10")
val_df = df_final.filter(F.col("pickup_month") == "2019-11")
test_df = df_final.filter(F.col("pickup_month") == "2019-12")


def save_data(df, name):
    path = f"{OUTPUT_PATH}/{name}"
    df.select("features", "fare_amount") \
      .write.mode("overwrite") \
      .parquet(path)
    print(f"Saved {name} to {path}")

save_data(train_df, "train_2019")
save_data(val_df, "val_2019")
save_data(test_df, "test_2019")


pipeline_model.write().overwrite().save(f"{BASE_PATH}/models/feature_pipeline_2019")

print("\n Feature Engineering Complete ")
df_final.select("features", "fare_amount").show(5, truncate=False)

Saved train_2019 to /content/drive/MyDrive/yellow_tripdata_2019/data/final/train_2019
Saved val_2019 to /content/drive/MyDrive/yellow_tripdata_2019/data/final/val_2019
Saved test_2019 to /content/drive/MyDrive/yellow_tripdata_2019/data/final/test_2019

 Feature Engineering Complete 
+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-----------+
|features                                                                                                                                                                                                                                                |fare_amount|
+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

Model Evaluation

In [36]:
# Load Processed Data for Model Evaluation
BASE_PATH = "/content/drive/MyDrive/yellow_tripdata_2019"

from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.ml.regression import GBTRegressionModel
from pyspark.sql.types import *
import numpy as np

if 'spark' in locals() and spark is not None:
    try:
        spark.stop()
    except Exception as e:
        print(f"Warning: Could not stop existing SparkSession: {e}")

spark = SparkSession.builder \
    .appName("NYC Taxi Prediction") \
    .config("spark.driver.memory", "8g") \
    .config("spark.executor.memory", "8g") \
    .getOrCreate()


train_df = spark.read.parquet(f"{BASE_PATH}/data/final/train_2019")
val_df = spark.read.parquet(f"{BASE_PATH}/data/final/val_2019")
test_df = spark.read.parquet(f"{BASE_PATH}/data/final/test_2019")

print(f"Train (Jan-Oct): {train_df.count():,} rows ({train_df.count()/(train_df.count()+val_df.count()+test_df.count())*100:.1f}%")
print(f"Val (Nov):       {val_df.count():,} rows")
print(f"Test (Dec):      {test_df.count():,} rows")

Train (Jan-Oct): 68,360,825 rows (83.8%
Val (Nov):       6,603,163 rows
Test (Dec):      6,617,319 rows


In [37]:
# Train Initial Random Forest Model
from pyspark.ml.regression import RandomForestRegressor

rf = RandomForestRegressor(
    featuresCol="features",
    labelCol="fare_amount",
    numTrees=3,
    maxDepth=3,
    subsamplingRate=0.7
)

best_model = rf.fit(train_df)

In [38]:
# Evaluate Initial Model Performance
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.sql import Row
from pyspark.sql.functions import col

print("DISTRIBUTED REGRESSION METRICS\n")


test_preds = best_model.transform(test_df).cache()


evaluators = {
    "RMSE": RegressionEvaluator(
        labelCol="fare_amount",
        predictionCol="prediction",
        metricName="rmse"
    ),
    "MSE": RegressionEvaluator(
        labelCol="fare_amount",
        predictionCol="prediction",
        metricName="mse"
    ),
    "MAE": RegressionEvaluator(
        labelCol="fare_amount",
        predictionCol="prediction",
        metricName="mae"
    ),
    "R2": RegressionEvaluator(
        labelCol="fare_amount",
        predictionCol="prediction",
        metricName="r2"
    )
}


metrics = {}

for name, evaluator in evaluators.items():
    score = evaluator.evaluate(test_preds)
    rounded_score = __builtins__.round(score, 4)
    metrics[name] = rounded_score
    print(f"{name:<6}: {rounded_score}")


print("\nSUMMARY TABLE:")
summary_df = spark.createDataFrame([Row(**metrics)])
summary_df.show(truncate=False)


test_preds.unpersist()

DISTRIBUTED REGRESSION METRICS

RMSE  : 5.5944
MSE   : 31.2977
MAE   : 3.1142
R2    : 0.7702

SUMMARY TABLE:
+------+-------+------+------+
|RMSE  |MSE    |MAE   |R2    |
+------+-------+------+------+
|5.5944|31.2977|3.1142|0.7702|
+------+-------+------+------+



DataFrame[features: vector, fare_amount: double, prediction: double]

In [39]:
# Analyze Temporal Performance Split
from pyspark.sql.functions import col, avg, lit

print("TEMPORAL SPLIT ANALYSIS:")
print("Temporal ordering confirmed:")
print("  Train: Jan-Oct 2015")
print("  Val:   Nov 2015")
print("  Test:  Dec 2015")


monthly_perf = (
    test_preds
        .withColumn("pickup_month", lit("2015-12"))
        .groupBy("pickup_month")
        .agg(
            avg(col("prediction") - col("fare_amount")).alias("mean_error"),
            avg(col("fare_amount")).alias("avg_fare")
        )
)


monthly_perf.show(truncate=False)

TEMPORAL SPLIT ANALYSIS:
Temporal ordering confirmed:
  Train: Jan-Oct 2015
  Val:   Nov 2015
  Test:  Dec 2015
+------------+--------------------+------------------+
|pickup_month|mean_error          |avg_fare          |
+------------+--------------------+------------------+
|2015-12     |-0.15553930824346893|13.293614698037121|
+------------+--------------------+------------------+



In [40]:
# Perform 5-Fold Stratified Cross-Validation
from pyspark.sql.functions import col, when
from pyspark.ml.evaluation import RegressionEvaluator
import numpy as np

print("5-FOLD STRATIFIED CROSS-VALIDATION")


fare_buckets = test_preds.withColumn(
    "fare_bucket",
    when(col("fare_amount") < 10, "low")
    .when(col("fare_amount") < 25, "medium")
    .otherwise("high")
)


print("Fare distribution:")
fare_buckets.groupBy("fare_bucket") \
    .count() \
    .orderBy("fare_bucket") \
    .show()


cv_evaluator = RegressionEvaluator(
    metricName="rmse",
    labelCol="fare_amount",
    predictionCol="prediction"
)

cv_scores = []


for i in range(5):

    fold = (
        fare_buckets
            .sample(withReplacement=False, fraction=0.2, seed=i)
            .select("features", "fare_amount")
            .cache()
    )

    fold_pred = best_model.transform(fold)

    rmse = cv_evaluator.evaluate(fold_pred)
    cv_scores.append(rmse)

    fold.unpersist()


print(f"Mean RMSE: ${np.mean(cv_scores):.2f} \u00b1 {np.std(cv_scores):.2f}")
print("CV Scores:", [f"{x:.2f}" for x in cv_scores])

5-FOLD STRATIFIED CROSS-VALIDATION
Fare distribution:
+-----------+-------+
|fare_bucket|  count|
+-----------+-------+
|       high| 690670|
|        low|3378344|
|     medium|2548305|
+-----------+-------+

Mean RMSE: $5.60 ± 0.03
CV Scores: ['5.63', '5.54', '5.59', '5.62', '5.60']


In [41]:
# Calculate Bootstrap Confidence Intervals for RMSE
from pyspark.ml.evaluation import RegressionEvaluator
import numpy as np

print("BOOTSTRAP CONFIDENCE INTERVALS")


test_preds_cached = (
    best_model
        .transform(test_df.coalesce(10))
        .select("prediction", "fare_amount")
        .cache()
)

test_preds_cached.count()


evaluator = RegressionEvaluator(
    labelCol="fare_amount",
    predictionCol="prediction",
    metricName="rmse"
)

def bootstrap_rmse_ci(n_bootstrap=50):

    rmse_scores = []

    for i in range(n_bootstrap):


        boot_sample = (
            test_preds_cached
                .sample(withReplacement=True, fraction=0.8, seed=i)
        )

        rmse = evaluator.evaluate(boot_sample)
        rmse_scores.append(rmse)


    rmse_scores = sorted(rmse_scores)

    lower_idx = int(0.025 * len(rmse_scores))
    upper_idx = int(0.975 * len(rmse_scores))

    ci_lower = rmse_scores[lower_idx]
    ci_upper = rmse_scores[upper_idx]

    return np.mean(rmse_scores), ci_lower, ci_upper



rmse_mean, rmse_ci_low, rmse_ci_high = bootstrap_rmse_ci(50)

print("RMSE Bootstrap Results:")
print(f"  Mean RMSE: ${rmse_mean:.2f}")
print(f"  95% CI:    ${rmse_ci_low:.2f} - ${rmse_ci_high:.2f}")
print(f"  CI Width:  ${rmse_ci_high - rmse_ci_low:.2f} (narrow = stable model)")


test_preds_cached.unpersist()

BOOTSTRAP CONFIDENCE INTERVALS
RMSE Bootstrap Results:
  Mean RMSE: $5.60
  95% CI:    $5.56 - $5.64
  CI Width:  $0.09 (narrow = stable model)


DataFrame[prediction: double, fare_amount: double]

In [42]:
# Align Model Performance with Business Metrics
from pyspark.sql.functions import col, avg, abs, count, when

print("BUSINESS METRICS ALIGNMENT")


test_with_profit = (
    test_preds
        .select("fare_amount", "prediction")
        .withColumn("driver_profit", col("fare_amount"))
        .withColumn("predicted_profit", col("prediction"))
)


business_metrics = (
    test_with_profit
        .agg(
            avg(col("driver_profit")).alias("avg_driver_profit"),
            avg(col("predicted_profit")).alias("avg_pred_profit"),
            avg(abs(col("driver_profit") - col("predicted_profit")))
                .alias("profit_forecast_error"),
            count(when(col("predicted_profit") > 0, True))
                .alias("profitable_trips_pred"),
            count(when(col("driver_profit") > 0, True))
                .alias("profitable_trips_actual")
        )
        .collect()[0]
)

print("NYC Taxi Driver Economics:")
print(f"Actual avg profit/trip:     ${business_metrics['avg_driver_profit']:.2f}")
print(f"Predicted avg profit/trip:  ${business_metrics['avg_pred_profit']:.2f}")
print(f"Profit forecast error:      ${business_metrics['profit_forecast_error']:.2f}")
print(f"Profitable trips actual:    {business_metrics['profitable_trips_actual']:,}")
print(f"Profitable trips predicted: {business_metrics['profitable_trips_pred']:,}")

BUSINESS METRICS ALIGNMENT
NYC Taxi Driver Economics:
Actual avg profit/trip:     $13.29
Predicted avg profit/trip:  $13.14
Profit forecast error:      $3.11
Profitable trips actual:    6,617,319
Profitable trips predicted: 6,617,319


Model training

In [43]:
# Load Data for Model Training and Comparison
BASE_PATH = "/content/drive/MyDrive/yellow_tripdata_2019"

from pyspark.sql import SparkSession
from pyspark.ml.regression import LinearRegression, GBTRegressor, RandomForestRegressor
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
import pyspark.sql.functions as F

spark = SparkSession.builder.getOrCreate()
print(f"Spark {spark.version} ready")

train_df = spark.read.parquet(f"{BASE_PATH}/data/final/train_2019")
val_df = spark.read.parquet(f"{BASE_PATH}/data/final/val_2019")
test_df = spark.read.parquet(f"{BASE_PATH}/data/final/test_2019")

print(f"Train: {train_df.count():,} rows | Schema: {train_df.columns}")
print(f"Val:   {val_df.count():,} rows")
print(f"Test:  {test_df.count():,} rows")
train_df.show(3, truncate=False)

Spark 3.5.0 ready
Train: 68,360,825 rows | Schema: ['features', 'fare_amount']
Val:   6,603,163 rows
Test:  6,617,319 rows
+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-----------+
|features                                                                                                                                                                                                                                                |fare_amount|
+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-----------+
|[1.1699727455587712,1.8139108170292353,0.0,0.1388096899939466,-2.3068364200405878,-0.76

In [44]:
# Define Regression Model Configurations
from pyspark.ml.regression import (
    LinearRegression,
    GBTRegressor,
    RandomForestRegressor
)


models_config = {

    "1. Linear Regression": LinearRegression(
        featuresCol="features",
        labelCol="fare_amount",
        maxIter=50,
        regParam=0.1,
        elasticNetParam=0.5
    ),

    "2. Gradient Boosted Trees": GBTRegressor(
        featuresCol="features",
        labelCol="fare_amount",
        maxIter=20,
        maxDepth=6,
        stepSize=0.05
    ),

    "3. Random Forest": RandomForestRegressor(
        featuresCol="features",
        labelCol="fare_amount",
        numTrees=20,
        maxDepth=8,
        featureSubsetStrategy="sqrt"
    ),

    "4. Ridge Regression": LinearRegression(
        featuresCol="features",
        labelCol="fare_amount",
        maxIter=50,
        regParam=1.0,
        elasticNetParam=0.0
    )
}

print("4 MLlib Regression Models:")
for name in models_config.keys():
    print(f"  • {name}")

4 MLlib Regression Models:
  • 1. Linear Regression
  • 2. Gradient Boosted Trees
  • 3. Random Forest
  • 4. Ridge Regression


In [45]:
# Train and Evaluate Multiple Models on Sampled Data
from pyspark.ml.evaluation import RegressionEvaluator


train_small = (
    train_df
        .select("features", "fare_amount")
        .sample(withReplacement=False, fraction=0.01, seed=42)
        .cache()
)

val_small = (
    val_df
        .select("features", "fare_amount")
        .sample(withReplacement=False, fraction=0.05, seed=42)
        .cache()
)

print(f"Train small: {train_small.count():,} rows")
print(f"Val small:   {val_small.count():,} rows")


evaluator_rmse = RegressionEvaluator(
    labelCol="fare_amount",
    predictionCol="prediction",
    metricName="rmse"
)

evaluator_r2 = RegressionEvaluator(
    labelCol="fare_amount",
    predictionCol="prediction",
    metricName="r2"
)

trained_models = {}
val_scores = []


for name, model in models_config.items():

    print(f"\nTraining {name}...")

    model_trained = model.fit(train_small)
    trained_models[name] = model_trained

    val_pred = model_trained.transform(val_small)

    rmse = evaluator_rmse.evaluate(val_pred)
    r2 = evaluator_r2.evaluate(val_pred)

    val_scores.append((name, __builtins__.round(rmse, 3), __builtins__.round(r2, 4)))

    print(f"RMSE: ${rmse:.2f} | R²: {r2:.4f}")


results_df = spark.createDataFrame(
    val_scores,
    ["Model", "RMSE", "R2"]
)

print("\nValidation Results (Sorted by RMSE):")
results_df.orderBy("RMSE").show(truncate=False)


train_small.unpersist()
val_small.unpersist()

Train small: 684,994 rows
Val small:   330,009 rows

Training 1. Linear Regression...
RMSE: $4.19 | R²: 0.8657

Training 2. Gradient Boosted Trees...
RMSE: $4.09 | R²: 0.8720

Training 3. Random Forest...
RMSE: $4.18 | R²: 0.8663

Training 4. Ridge Regression...
RMSE: $4.27 | R²: 0.8605

Validation Results (Sorted by RMSE):
+-------------------------+-----+------+
|Model                    |RMSE |R2    |
+-------------------------+-----+------+
|2. Gradient Boosted Trees|4.093|0.872 |
|3. Random Forest         |4.183|0.8663|
|1. Linear Regression     |4.193|0.8657|
|4. Ridge Regression      |4.274|0.8605|
+-------------------------+-----+------+



DataFrame[features: vector, fare_amount: double]

In [46]:
# Final Evaluation of Models on Test Data
from pyspark.ml.evaluation import RegressionEvaluator


test_data = test_df.select("features", "fare_amount")


evaluator_rmse = RegressionEvaluator(
    labelCol="fare_amount",
    predictionCol="prediction",
    metricName="rmse"
)

evaluator_mae = RegressionEvaluator(
    labelCol="fare_amount",
    predictionCol="prediction",
    metricName="mae"
)

evaluator_r2 = RegressionEvaluator(
    labelCol="fare_amount",
    predictionCol="prediction",
    metricName="r2"
)

print(f"{'Model':<25} {'Test RMSE ($)':<15} {'Test MAE ($)':<15} {'Test R²':<10}")

test_results = {}

for name, model in trained_models.items():

    print(f"\nTest predictions: {name}")

    test_pred = model.transform(test_data)

    rmse = evaluator_rmse.evaluate(test_pred)
    mae = evaluator_mae.evaluate(test_pred)
    r2 = evaluator_r2.evaluate(test_pred)

    test_results[name] = {
        "Test_RMSE": __builtins__.round(rmse, 3),
        "Test_MAE": __builtins__.round(mae, 3),
        "Test_R2": __builtins__.round(r2, 4)
    }

    print(f"  RMSE: ${rmse:.2f} | MAE: ${mae:.2f} | R²: {r2:.4f}")


print("\nFINAL RANKING:")

sorted_results = sorted(
    test_results.items(),
    key=lambda x: x[1]["Test_RMSE"]
)

for i, (name, scores) in enumerate(sorted_results, start=1):
    print(
        f"{i}. {name:<23} "
        f"RMSE=${scores['Test_RMSE']:6.2f}  "
        f"MAE=${scores['Test_MAE']:6.2f}  "
        f"R²={scores['Test_R2']:7.4f}"
    )

best_model_name = sorted_results[0][0]
print(f"\nBEST MODEL: {best_model_name}")

Model                     Test RMSE ($)   Test MAE ($)    Test R²   

Test predictions: 1. Linear Regression
  RMSE: $4.31 | MAE: $2.15 | R²: 0.8635

Test predictions: 2. Gradient Boosted Trees
  RMSE: $4.13 | MAE: $1.03 | R²: 0.8745

Test predictions: 3. Random Forest
  RMSE: $4.23 | MAE: $1.20 | R²: 0.8685

Test predictions: 4. Ridge Regression
  RMSE: $4.39 | MAE: $2.23 | R²: 0.8582

FINAL RANKING:
1. 2. Gradient Boosted Trees RMSE=$  4.13  MAE=$  1.03  R²= 0.8745
2. 3. Random Forest        RMSE=$  4.23  MAE=$  1.20  R²= 0.8685
3. 1. Linear Regression    RMSE=$  4.31  MAE=$  2.15  R²= 0.8635
4. 4. Ridge Regression     RMSE=$  4.39  MAE=$  2.23  R²= 0.8582

BEST MODEL: 2. Gradient Boosted Trees


In [ ]:
# Hyperparameter Tuning with Cross-Validation
from pyspark.ml.regression import GBTRegressor, RandomForestRegressor, LinearRegression
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.ml.evaluation import RegressionEvaluator

print(f"\nTUNING {best_model_name}...")


if "Gradient Boosted Trees" in best_model_name:
    model_tune = GBTRegressor(
        featuresCol="features",
        labelCol="fare_amount"
    )

    paramGrid = (
        ParamGridBuilder()
            .addGrid(model_tune.maxDepth, [3, 5])
            .addGrid(model_tune.maxIter, [10, 20])
            .build()
    )

elif "Random Forest" in best_model_name:
    model_tune = RandomForestRegressor(
        featuresCol="features",
        labelCol="fare_amount"
    )

    paramGrid = (
        ParamGridBuilder()
            .addGrid(model_tune.maxDepth, [4, 6])
            .addGrid(model_tune.numTrees, [10, 20])
            .build()
    )

else:
    model_tune = LinearRegression(
        featuresCol="features",
        labelCol="fare_amount"
    )

    paramGrid = (
        ParamGridBuilder()
            .addGrid(model_tune.regParam, [0.01, 0.1])
            .addGrid(model_tune.maxIter, [50, 100])
            .build()
    )


train_cv_sample = (
    train_df
        .select("features", "fare_amount")
        .sample(withReplacement=False, fraction=0.01, seed=42)
        .cache()
)

train_cv_sample.count()


evaluator = RegressionEvaluator(
    labelCol="fare_amount",
    predictionCol="prediction",
    metricName="rmse"
)


cv = CrossValidator(
    estimator=model_tune,
    estimatorParamMaps=paramGrid,
    evaluator=evaluator,
    numFolds=2
)

print("Running 2-Fold Cross-Validation on sampled data...")

cv_model = cv.fit(train_cv_sample)


test_data = test_df.select("features", "fare_amount")
cv_test_pred = cv_model.bestModel.transform(test_data)

cv_rmse = evaluator.evaluate(cv_test_pred)

print("\nBest Parameters:")
print(cv_model.bestModel.extractParamMap())

print(f"\nTuned Test RMSE: ${cv_rmse:.2f}")


train_cv_sample.unpersist()


TUNING 2. Gradient Boosted Trees...
Running 2-Fold Cross-Validation on sampled data...


In [21]:
# Analyze Feature Importance from Tree-based Model
best_tree_model = None
best_tree_model_name = None


for name, model in trained_models.items():
    if "Gradient Boosted Trees" in name or "Random Forest" in name:
        best_tree_model = model
        best_tree_model_name = name
        break


if best_tree_model and hasattr(best_tree_model, "featureImportances"):

    importances = best_tree_model.featureImportances.toArray()


    features = [
        "passenger", "trip_dist", "haversine", "duration",
        "hour", "peak_hr", "ratecode", "short_trip",
        "congestion", "airport"
    ]

    print(f"\nFEATURE IMPORTANCE ({best_tree_model_name}):")
    print(f"{'Feature':<15} {'Importance'}")

    for feat, imp in zip(features, importances):
        print(f"{feat:<15} {imp:.4f}")

else:
    print("\nSelected model is Linear Regression - no feature importance available.")


FEATURE IMPORTANCE (2. Gradient Boosted Trees):
Feature         Importance
passenger       0.0025
trip_dist       0.7845
haversine       0.0000
duration        0.1845
hour            0.0203
peak_hr         0.0002
ratecode        0.0006
short_trip      0.0012
congestion      0.0011
airport         0.0002


In [22]:
# Perform Residual Analysis
from pyspark.sql import functions as F


test_data = test_df.select("features", "fare_amount")


best_model = trained_models[best_model_name]


test_final = best_model.transform(test_data)


test_final = test_final.withColumn(
    "residual",
    F.col("fare_amount") - F.col("prediction")
)


test_final.select(
    "prediction",
    "fare_amount",
    "residual"
).show(10, truncate=False)


residual_summary = test_final.select(
    F.round(F.avg("residual"), 2).alias("mean_residual"),
    F.round(F.stddev("residual"), 2).alias("residual_std")
)

print("\nResidual Summary Statistics:")
residual_summary.show()

+------------------+-----------+--------------------+
|prediction        |fare_amount|residual            |
+------------------+-----------+--------------------+
|14.380399568256818|13.0       |-1.3803995682568182 |
|4.826534316443225 |5.0        |0.17346568355677494 |
|14.628142058969045|13.5       |-1.1281420589690452 |
|4.529649770010345 |4.5        |-0.0296497700103453 |
|18.061971121948112|16.5       |-1.5619711219481118 |
|6.2399438127303375|6.0        |-0.2399438127303375 |
|5.727786248917669 |5.5        |-0.22778624891766874|
|9.762671149380935 |9.5        |-0.2626711493809353 |
|7.358449020762882 |7.5        |0.14155097923711768 |
|5.313423052234156 |5.5        |0.1865769477658441  |
+------------------+-----------+--------------------+
only showing top 10 rows


Residual Summary Statistics:
+-------------+------------+
|mean_residual|residual_std|
+-------------+------------+
|         0.05|        4.13|
+-------------+------------+



In [23]:
# Save Best Models and Export Results for Visualization
from pyspark.sql import functions as F


safe_model_name = best_model_name.replace(" ", "_").lower()


best_model.write().overwrite().save(
    f"{BASE_PATH}/tests/{safe_model_name}"
)


cv_model.bestModel.write().overwrite().save(
    f"{BASE_PATH}/tests/tuned_best_model"
)


test_data = test_df.select("features", "fare_amount")


test_final = best_model.transform(test_data)


test_final_with_residual = test_final.withColumn(
    "residual",
    F.col("fare_amount") - F.col("prediction")
)


(
    test_final_with_residual
        .select("prediction", "fare_amount", "residual")
        .coalesce(1)
        .write
        .mode("overwrite")
        .option("header", "true")
        .csv(f"{BASE_PATH}/tableau/final_predictions")
)


print(f"Best model saved at: {BASE_PATH}/tests/{safe_model_name}")
print(f"Tuned model saved at: {BASE_PATH}/tests/tuned_best_model")
print(f"Tableau export saved at: {BASE_PATH}/tableau/final_predictions")


print("\nBEST RESULTS:")
if best_model_name in test_results:
    scores = test_results[best_model_name]
    print(
        f"{best_model_name}: "
        f"RMSE=${scores['Test_RMSE']:.2f}, "
        f"R²={scores['Test_R2']:.4f}"
    )

Best model saved at: /content/drive/MyDrive/yellow_tripdata_2019/tests/2._gradient_boosted_trees
Tuned model saved at: /content/drive/MyDrive/yellow_tripdata_2019/tests/tuned_best_model
Tableau export saved at: /content/drive/MyDrive/yellow_tripdata_2019/tableau/final_predictions

BEST RESULTS:
2. Gradient Boosted Trees: RMSE=$4.13, R²=0.8745


In [24]:
# Export Model Metrics to CSV for Tableau
from pyspark.sql import SparkSession
from pyspark.ml.evaluation import RegressionEvaluator
import pandas as pd
from pyspark.ml.regression import GBTRegressionModel
spark = SparkSession.builder.getOrCreate()


BASE_PATH = "/content/drive/MyDrive/yellow_tripdata_2019"


best_model_name = "2. Gradient Boosted Trees"


if 'trained_models' in globals() and best_model_name in trained_models:
    best_model = trained_models[best_model_name]
else:
    print(f"Warning: 'trained_models' or '{best_model_name}' not found in current scope. Attempting to load best model from disk.")

    safe_model_name = best_model_name.replace(" ", "_").lower()
    model_path = f"{BASE_PATH}/tests/{safe_model_name}"
    try:

        best_model = GBTRegressionModel.load(model_path)
        print(f"Successfully loaded '{best_model_name}' from {model_path}")
    except Exception as e:
        raise RuntimeError(f"Could not load best_model from disk at {model_path}. Ensure previous cells were executed to train and save the model, or check the path. Error: {e}")


test_df = spark.read.parquet(f"{BASE_PATH}/data/final/test_2019")

evaluator_rmse = RegressionEvaluator(labelCol="fare_amount", metricName="rmse")
evaluator_mae = RegressionEvaluator(labelCol="fare_amount", metricName="mae")
evaluator_r2 = RegressionEvaluator(labelCol="fare_amount", metricName="r2")

predictions = best_model.transform(test_df)

rmse = evaluator_rmse.evaluate(predictions)
mae = evaluator_mae.evaluate(predictions)
r2 = evaluator_r2.evaluate(predictions)

metrics_df = pd.DataFrame({
    "Model": ["Gradient Boosted Trees"],
    "RMSE": [rmse],
    "MAE": [mae],
    "R2": [r2]
})

metrics_df.to_csv(f"{BASE_PATH}/tableau/model_comparison.csv", index=False)

In [25]:
# Export Feature Importance to CSV for Tableau
import pandas as pd


BASE_PATH = "/content/drive/MyDrive/yellow_tripdata_2019"


importances = best_model.featureImportances.toArray()


numeric_features_names = [
    "passenger_count",
    "trip_distance",
    "haversine_km",
    "trip_duration_min",
    "pickup_hour",
    "is_peak_hour",
    "is_weekend"
]


one_hot_encoded_dow_names = [f"dow_encoded_{i}" for i in range(6)]


feature_names = numeric_features_names + one_hot_encoded_dow_names


if len(feature_names) != len(importances):
    print(f"Warning: Mismatch between reconstructed feature names count ({len(feature_names)}) and actual importances count ({len(importances)}). Using generic names as fallback.")
    feature_names = [f"feature_{i}" for i in range(len(importances))]


fi_df = pd.DataFrame({
    "Feature": feature_names,
    "Importance": importances
})

fi_df = fi_df.sort_values("Importance", ascending=False)


fi_df.to_csv(f"{BASE_PATH}/tableau/feature_importance.csv", index=False)

In [26]:
# Export Data Quality Summary to CSV for Tableau
from pyspark.sql.functions import col, count, when
import pandas as pd

total_rows = test_df.count()

null_counts = [
    (c, test_df.filter(col(c).isNull()).count())
    for c in test_df.columns
]

dq_df = pd.DataFrame(null_counts, columns=["Column", "Null_Count"])
dq_df["Null_Percentage"] = dq_df["Null_Count"] / total_rows * 100


dq_df.to_csv(f"{BASE_PATH}/tableau/data_quality_summary.csv", index=False)

In [27]:
# Export Final Predictions to CSV for Tableau
predictions.select("fare_amount", "prediction") \
    .toPandas() \
    .to_csv(f"{BASE_PATH}/tableau/final_predictions.csv", index=False)